In [1]:
import os
os.environ["HADOOP_USER_NAME"] = "root"

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField , StructType , IntegerType , StringType , DateType , DoubleType
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('nti_project') \
        .config("spark.jars.packages", "net.snowflake:spark-snowflake_2.12:2.15.0-spark_3.4,net.snowflake:snowflake-jdbc:3.14.4") \
        .getOrCreate()

print("✅ Spark started successfully!")
print(f"Version: {spark.version}")
spark.sparkContext.setLogLevel("WARN")

✅ Spark started successfully!
Version: 3.5.0


In [4]:
import configparser

config = configparser.ConfigParser()
config.read("/home/jovyan/work/connections.my_example_connection")

['/home/jovyan/work/connections.my_example_connection']

In [5]:
sfURL = config['connections.my_example_connection']['account'].strip('"')
sfUser = config['connections.my_example_connection']['user'].strip('"')
sfPassword = config['connections.my_example_connection']['password'].strip('"')
sfRole = config['connections.my_example_connection']['role'].strip('"')  
sfDatabase = config['connections.my_example_connection']['database'].strip('"')
sfSchema = config['connections.my_example_connection']['schema'].strip('"')
sfWarehouse = config['connections.my_example_connection']['warehouse'].strip('"')

In [6]:
sfOptions = {
    "sfURL": sfURL,
    "sfUser": sfUser,
    "sfPassword": sfPassword,
    "sfDatabase": sfDatabase,
    "sfSchema": sfSchema,
    "sfWarehouse": sfWarehouse,
}

In [7]:
base_path = "hdfs://hadoop-namenode:9000/Data/silver/"

In [8]:
def load_table(table_name, mode="overwrite"):
    print(f"\n📤 Loading {table_name}...")
    df = spark.read.parquet(f"{base_path}/{table_name}")
    count = df.count()
    print(f" Records: {count}")
    
    df.write \
        .format("snowflake") \
        .options(**sfOptions) \
        .option("dbtable", table_name.upper()) \
        .mode(mode) \
        .save()
    
    print(f"✅ {table_name} loaded")

In [9]:
try:
    load_table("dim_categories", "overwrite")
    load_table("dim_customers", "overwrite")
    load_table("dim_orders", "overwrite")
    load_table("dim_ordersdetails", "overwrite")
    load_table("dim_product", "overwrite")     
    load_table("dim_employees", "overwrite")
    load_table("dim_suppliers", "overwrite")
    load_table("dim_shippers", "overwrite")
except Exception as e:
    print(f"\n❌ LOADING FAILED: {e}")


📤 Loading dim_categories...
 Records: 8
✅ dim_categories loaded

📤 Loading dim_customers...
 Records: 91
✅ dim_customers loaded

📤 Loading dim_orders...
 Records: 830
✅ dim_orders loaded

📤 Loading dim_ordersdetails...
 Records: 2155
✅ dim_ordersdetails loaded

📤 Loading dim_product...
 Records: 77
✅ dim_product loaded

📤 Loading dim_employees...
 Records: 9
✅ dim_employees loaded

📤 Loading dim_suppliers...
 Records: 29
✅ dim_suppliers loaded

📤 Loading dim_shippers...
 Records: 3
✅ dim_shippers loaded
